# TDNEGF Current Workflow Guide (Block Backend)

This notebook documents the **current, recommended TDNEGF workflow** in this repository.
It is implementation-faithful to the code in `src/` and the active examples in `examples/`.

> Source-of-truth priority used here:
> 1. `README.md`
> 2. `examples/01_two_terminal_square_lattice.jl` and `examples/04_blocks_1d_twochannel_benchmark.jl`
> 3. `src/` implementation details (`eom_tdnegf_blocks!`, `ExperimentalBlockRHSParams`, `SelfEnergyBlock`, observable APIs)
>
> `test/TDNGF_V3.ipynb` is treated as **legacy reference only**.

## Working plan used for this notebook

1. Identify the active workflow from README + examples + current `src` API.
2. Cross-check legacy `test/TDNGF_V3.ipynb` for mismatches.
3. Build one coherent, runnable notebook around the **block-based backend**:
   - architecture and data layout,
   - initialization,
   - propagation,
   - observables,
   - practical audit notes.
4. Keep any changes minimal and avoid refactoring package APIs.

## 1) Architecture overview (current)

The repository currently ships two backends:

- **Recommended/current**: block-based heterogeneous auxiliary backend
  - RHS: `eom_tdnegf_blocks!`
  - params: `ExperimentalBlockRHSParams`
  - static block object: `SelfEnergyBlock`
- **Legacy compatibility**: rectangular auxiliary backend
  - RHS: `eom_tdnegf!`
  - params: `ModelParamsTDNEGF`

In current examples, the block backend is the main path.

### Practical flow

1. Build a `ModelParamsTDNEGF` (`p_model`) for geometry + base metadata.
2. Build `H_ab`, self-energy coefficients (`Σᴸ`, `Σᴳ`, `χ`), and lead couplings (`ξ`).
3. Create one or more `SelfEnergyBlock` objects.
4. Create `ExperimentalBlockRHSParams(H_ab, blocks, Δ_blocks, p_model)`.
5. Build initial state vector `u0` from `dims_ρ_ab` and `aux_layout.total_size`.
6. Solve `ODEProblem(eom_tdnegf_blocks!, u0, tspan, p_blocks)`.
7. Post-process via `pointer_blocks(...)` + observable functions.

In [ ]:
using Pkg
Pkg.activate(joinpath(pwd()))

using TDNEGF
using DifferentialEquations
using LinearAlgebra
using Statistics

## 2) Build a minimal current-workflow setup

This cell follows the same implementation pattern as `examples/01_two_terminal_square_lattice.jl`,
but uses a shorter run configuration so it is easier to execute interactively.

**What this step prepares:**
- `p_model`: model geometry/index metadata,
- `p_blocks`: block backend RHS params and preallocations,
- `u0`: flattened initial ODE state matching current block layout.

In [ ]:
function init_params_blocks_demo(; Nx::Int=6, Ny::Int=2, Nσ::Int=2, N_orb::Int=1,
                                  N_λ1::Int=49, N_λ2::Int=20,
                                  β::Float64=33.0, γ::Float64=1.0, γso::ComplexF64=0.5 + 0im,
                                  Vbias::Float64=1.0)
    # Pole data loaded from repository data files through the package helper.
    Rλ, zλ = load_poles_square(N_λ1, N_λ2)

    # Geometry/model metadata container (also reused by observable helpers).
    p_model = ModelParamsTDNEGF(Nx=Nx, Ny=Ny, Nσ=Nσ, N_orb=N_orb,
                                Nα=2, N_λ1=N_λ1, N_λ2=N_λ2)

    # Device Hamiltonian.
    H_ab = build_H_ab(; Nx=p_model.Nx, Ny=p_model.Ny, Nσ=p_model.Nσ,
                      N_orb=p_model.N_orb, γ=γ, γso=γso)
    p_model.H_ab .= H_ab
    p_model.H0_ab .= H_ab

    # Lead self-energy ingredients (same helper API as current examples).
    ΣL_nλ = build_Σᴸ_nλ(Rλ, zλ, p_model.Ny, p_model.Nσ, p_model.N_orb,
                       p_model.N_λ1, p_model.N_λ2; β=β, γ=γ)
    ΣG_nλ = build_Σᴳ_nλ(Rλ, zλ, p_model.Ny, p_model.Nσ, p_model.N_orb,
                       p_model.N_λ1, p_model.N_λ2; β=β, γ=γ)
    χ_nλ  = build_χ_nλ(zλ, p_model.Ny, p_model.Nσ, p_model.N_orb,
                      p_model.N_λ1, p_model.N_λ2; β=β, γ=γ)

    # Left/right couplings.
    ξ_anL = build_ξ_an(p_model.Nx, p_model.Ny, p_model.Nσ, p_model.N_orb;
                       xcol=1, y_coup=1:p_model.Ny)
    ξ_anR = build_ξ_an(p_model.Nx, p_model.Ny, p_model.Nσ, p_model.N_orb;
                       xcol=p_model.Nx, y_coup=1:p_model.Ny)

    # Static block definitions.
    left_block  = SelfEnergyBlock(:left,  p_model.Nc, p_model.N_λ1, p_model.N_λ2,
                                  ΣL_nλ, ΣG_nλ, χ_nλ, ξ_anL)
    right_block = SelfEnergyBlock(:right, p_model.Nc, p_model.N_λ1, p_model.N_λ2,
                                  ΣL_nλ, ΣG_nλ, χ_nλ, ξ_anR)
    blocks = [left_block, right_block]

    # Dynamic lead shifts kept at RHS-params level for easy bias scans.
    Δ_blocks = ComplexF64[+Vbias/2 + 0im, -Vbias/2 + 0im]

    # Preferred constructor includes observable geometry metadata from p_model.
    p_blocks = ExperimentalBlockRHSParams(p_model.H_ab, blocks, Δ_blocks, p_model)

    # Flattened initial state: [vec(ρ); vec(aux-block sectors)].
    u0 = zeros(ComplexF64, p_blocks.dims_ρ_ab[1]^2 + p_blocks.aux_layout.total_size)

    return p_model, p_blocks, u0
end

p_model, p_blocks, u0 = init_params_blocks_demo();

println("Ns = ", p_model.Ns)
println("N_sites = ", p_model.N_sites)
println("N_blocks = ", length(p_blocks.blocks))
println("state length = ", length(u0))
println("ρ size = ", p_blocks.dims_ρ_ab)
println("aux total size = ", p_blocks.aux_layout.total_size)

## 3) Main data structures (current implementation)

This is the minimum set to understand/debug the block workflow:

- `ModelParamsTDNEGF`
  - still central for geometry, dimensions, and local observable metadata.
- `SelfEnergyBlock`
  - static block data (`Nc`, `N_λ1`, `N_λ2`, `ΣL/ΣG/χ`, `ξ`).
- `ExperimentalBlockRHSParams`
  - dynamic RHS context: `H_ab`, `Δ_blocks`, preallocated buffers, layout metadata.
- `pointer_blocks(u, dims_ρ_ab, aux_layout)`
  - typed views into the flattened state (`ρ`, block `Ψ`, pair `Ω` sectors).

The split is intentional: static per-block structure in `SelfEnergyBlock`, runtime-evolving/problem-specific state in `ExperimentalBlockRHSParams`.

In [ ]:
println(typeof(p_model))
println(typeof(p_blocks))
println(typeof(p_blocks.blocks[1]))

ptr0 = pointer_blocks(u0, p_blocks.dims_ρ_ab, p_blocks.aux_layout)
println(typeof(ptr0))
println("ρ view size: ", size(ptr0.ρ_ab))
println("first block Ψ size: ", size(ptr0.blocks[1].Ψ_anλ))

## 4) Propagation workflow

This is the core execution path used in current examples.

**What this step does:**
- builds `ODEProblem` with `eom_tdnegf_blocks!`,
- integrates a short time window,
- stores snapshots for post-processing.

**Execution note:** this is usually runnable as-is in the repository environment, but runtime scales with system size and pole count.

In [ ]:
tspan = (0.0, 5.0)
prob = ODEProblem(eom_tdnegf_blocks!, u0, tspan, p_blocks)

sol = solve(prob, Vern7();
            saveat=0.5,
            adaptive=true,
            dense=false,
            reltol=1e-6,
            abstol=1e-8)

println("saved steps: ", length(sol.t))
println("final time: ", sol.t[end])

## 5) Observables and post-processing

Current observable workflow for block backend:

1. Allocate `ObservablesTDNEGF` with `N_leads = length(p_blocks.blocks)`.
2. For each saved state `ut`, build `ptr = pointer_blocks(ut, ...)`.
3. Call:
   - `obs_n_i!(ptr, p_model, obs)` for local charge,
   - `obs_σ_i!(ptr, p_model, obs)` for local electron spin,
   - `obs_Ixα!(ptr, p_blocks, obs)` for block-indexed charge/spin currents.

Important indexing note: in block mode, current index `α` follows **block order**.

In [ ]:
obs = ObservablesTDNEGF(p_model; N_tmax=length(sol.t), N_leads=length(p_blocks.blocks))
obs.t .= sol.t

for (it, ut) in enumerate(sol.u)
    obs.idx = it
    ptr = pointer_blocks(ut, p_blocks.dims_ρ_ab, p_blocks.aux_layout)

    # Local (backend-independent in practice: depends on ρ_ab).
    obs_n_i!(ptr, p_model, obs)
    obs_σ_i!(ptr, p_model, obs)

    # Block-native lead/block current path.
    obs_Ixα!(ptr, p_blocks, obs)
end

println("obs.n_i shape   = ", size(obs.n_i))
println("obs.σx_i shape  = ", size(obs.σx_i))
println("obs.Iα shape    = ", size(obs.Iα))
println("obs.Iαx shape   = ", size(obs.Iαx))

In [ ]:
# Quick sanity summaries useful for debugging/maintenance.
charge_total_t = vec(sum(obs.n_i; dims=1))
I_left = obs.Iα[1, :]
I_right = obs.Iα[2, :]

println("total charge (first, last) = ", (charge_total_t[1], charge_total_t[end]))
println("I_left  (first, last) = ", (I_left[1], I_left[end]))
println("I_right (first, last) = ", (I_right[1], I_right[end]))

## 6) Legacy vs current workflow (practical mapping)

### What remained the same

- Physical one-time TDNEGF structure (`ρ`, auxiliary sectors, ODE propagation).
- Pole-based self-energy ingredient construction.
- Observable concepts (local density/spin + lead currents).

### What changed in current recommended path

- **Primary RHS**: from legacy `eom_tdnegf!` to `eom_tdnegf_blocks!`.
- **Auxiliary layout**: from fixed rectangular tensors to heterogeneous block/pair layout.
- **Current indexing meaning**: in block mode, `obs.Iα` indexes blocks, not implicitly legacy-lead slots.
- **Parameter split**: static block metadata in `SelfEnergyBlock`, dynamic shifts in `Δ_blocks` inside `ExperimentalBlockRHSParams`.

### Steps reorganized

- Legacy notebook style (`test/TDNGF_V3.ipynb`) mixed many definitions, experiments, and plotting in one long narrative.
- Current workflow is cleaner when separated into:
  1. parameter/model construction,
  2. block assembly,
  3. propagation,
  4. post-processing.

### No longer recommended as main workflow

- Treating large monolithic notebook code as the source of truth for core APIs.
- Building new studies around ad-hoc legacy EOM variants when block backend is available.

## 6.1) Old-to-new implementation mapping (concrete)

| Legacy notebook pattern (`test/TDNGF_V3.ipynb`) | Current recommended workflow | Practical action for maintainers |
|---|---|---|
| In-notebook definitions for model/self-energy/EOM/observables | Use package API from `src/` and active examples | Do **not** copy old internal function bodies into new studies; import from `TDNEGF` instead. |
| Single monolithic flow mixing transport, semiclassical-spin experiments, and plotting | Staged workflow: setup → blocks → solve → observables | Keep transport baseline reproducible first; add side experiments in separate notebooks/scripts. |
| Rectangular auxiliary emphasis (`eom_tdnegf!`, legacy pointer layout) | Heterogeneous block auxiliary emphasis (`eom_tdnegf_blocks!`, `pointer_blocks`) | Default to block backend unless reproducing historical results. |
| Implicit lead-current indexing assumptions | Block-order indexing for `obs.Iα` / `obs.Iαx` | Explicitly document block ordering in each project and relabel when aggregating by physical lead. |

### Concepts that remained the same
- One-time TDNEGF ODE evolution using reduced density matrix + auxiliaries.
- Pole-expanded embedding self-energy ingredients (`Σ`, `χ`, `ξ`) as core inputs.
- Post-processing around local density/spin plus lead/block currents.

## 6.2) Verification checklist against current repository workflow

This notebook was written to match what is **actively used now**:

- `README.md` names the block backend as recommended and lists active example entry points.
- `examples/01_two_terminal_square_lattice.jl` uses `SelfEnergyBlock`, `ExperimentalBlockRHSParams`, `eom_tdnegf_blocks!`, and `pointer_blocks` observable flow.
- `examples/04_blocks_1d_twochannel_benchmark.jl` reinforces the same block-native solve and post-processing pattern.
- `src/eom_tdnegf.jl` defines `ExperimentalBlockRHSParams` and `eom_tdnegf_blocks!` as the current block-RHS implementation.
- `src/observables.jl` provides block-native `obs_Ixα!(ptr, p_blocks, obs)` and documents block-order indexing behavior.

### Ambiguity noted and resolution
- The requested legacy reference name appears as `TDNEGF_V3` in task text, while repository file is `test/TDNGF_V3.ipynb` (missing `E`).
- This notebook uses the on-disk file path as the operational legacy reference and treats it as non-authoritative for current APIs.

## 6.3) Executability discipline and honest run constraints

- The main code path is designed to run in a standard Julia environment with this repository activated.
- Required local assets: pole data loaded via `load_poles_square(...)` from repository `data/`.
- Runtime sensitivity: solve cost grows rapidly with system size and pole count; demo defaults are intentionally moderate.
- If running in non-Julia environments (CI shells without `julia` binary), treat code cells as reference templates and execute in a proper Julia runtime.

## 7) Light audit notes (legacy notebook vs current code)

### Observed mismatches / stale patterns

- `test/TDNGF_V3.ipynb` contains many in-notebook redefinitions of core functions (`build_Σ...`, `eom!`, observable structs), which now exist in package `src/` and should not be duplicated.
- The legacy notebook interleaves semiclassical-spin experiments and multiple testbeds with the transport baseline, which obscures the canonical solver path.
- Current repository examples clearly favor block backend setup (`SelfEnergyBlock` + `ExperimentalBlockRHSParams` + `eom_tdnegf_blocks!`).

### Minimal-fix policy applied here

- No broad refactor was applied to package source.
- A **new notebook guide** is added to document the current path faithfully.
- Legacy notebook remains untouched as historical reference.

### Maintenance pitfalls to watch

1. **Index meaning drift**: block-order current indexing can be misread as physical-lead indexing.
2. **Constructor choice**: using `ExperimentalBlockRHSParams(H_ab, blocks, Δ_blocks, p_model)` is preferred for current observable APIs.
3. **Hidden runtime assumptions**: pole counts and system size strongly affect solve cost; keep demo sizes small while debugging.
4. **API/documentation skew**: if examples change, update this guide and docs together to avoid stale onboarding paths.